# Divergent: conflicting traits and cross-franchise overlap

Owner: Thais. All the code lives in `experiments/divergent.py` (methods in `sorter.py`); this notebook only runs it and shows the outputs.

**Question:** when the traits point to two (or three) factions, does BERT pick one or split its probability? And does it map the HP houses onto the factions consistently?

Design decisions and the pre-registered word lists: `DATA_LOG.md` (entry "Divergent tests built").

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'experiments'))
import divergent as dv

dv.build()      # data/divergent/tests.csv (canonical rows appear once traits are filled)
dv.run()        # cached: only new rows are scored
dv.analyze()    # results/divergent/

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((dv.OUT / 'SUMMARY.md').read_text()))
for f in ('fig_entropy.png', 'fig_correspondence.png'):
    if (dv.OUT / f).exists():
        display(Image(filename=str(dv.OUT / f)))

## What I found

**Short answer:** BERT does not really read personality. It reacts to key words, it picks one faction instead of splitting when traits conflict, and it is only unsure when no faction has a matching word.

1. **Only some methods even get the easy cases.** On the 11 single-faction inputs, the fill-in-the-blank BERT (MLM) is at chance (21-24% vs. 20%) and answers "Dauntless" to more than half of *all* inputs. So for the MLM, the ambiguity numbers below describe a label prior, not the traits. Zero-shot NLI with the faction descriptions is the only strong method (82% on clear inputs).
2. **Conflicting traits: it picks one, it does not split.** With NLI + descriptions, 75% of the probability lands on the two mixed factions (chance 40%) and the top answer is one of them 95% of the time. But the split is lopsided (min/max ratio 0.27), and the two factions are the top two only 47% of the time. Entropy is higher on conflicting and three-faction inputs than on clear ones (+0.13 to +0.18 for NLI and embedding with descriptions), but no difference survives the Holm correction.
3. **Word order changes the answer.** The same two traits in reverse order shift the probability by about 3-4 points (NLI p = 0.001, embedding p = 0.009). A reader who understood the description would not care about the order.
4. **Divergent mode (three factions).** With NLI + descriptions, 87% of the probability lands on the three mixed factions (chance 60%). For Tris's aptitude result (selfless + brave + intelligent), 86% lands on Abnegation, Dauntless and Erudite. In the 4 book characters, one word dominates: "smart" sends Tris, Eric and Uriah to Erudite (68-72%). The source only describes personality for Dauntless characters, so the character test is a case study, not a measure.
5. **Factions -> houses.** Most methods look consistent only because they collapse onto one faction and one house (MLM: 96% "consistent", exactly its permutation baseline). Only embedding and NLI with descriptions beat the shuffled baseline (0.41 vs. 0.24 and 0.36 vs. 0.25, p < 0.01), e.g. Dauntless -> Gryffindor (0.67 in the embedding). Villains are the clearest case: Peter and Eric go to Slytherin (78-91%), but among the factions, where no group means "cruel", Peter is almost uniform (entropy 0.98).

**Limitations:** the probes are hand-written (74 inputs, pre-registered in DATA_LOG.md), so n is small; the character test has only 4 rows, all Dauntless; the comparison panel is simulated, not human.